In [ ]:
%pip install uv -q

In [ ]:
!uv init --bare

In [ ]:
!apt-get -qq -y install espeak-ng > /dev/null 2>&1

In [ ]:
!uv pip install --system -q kokoro soundfile librosa matplotlib pandas

In [ ]:
from kokoro import KPipeline
from IPython.display import display, Audio
import os
import soundfile as sf
import pandas
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import librosa

In [ ]:
pipeline = KPipeline(lang_code="a", repo_id='hexgrad/Kokoro-82M')

In [ ]:
OUTPUT_DIR = "wakeword_exploration"
os.makedirs(f"{OUTPUT_DIR}/positive", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/negative", exist_ok=True)

In [ ]:
WAKE_WORD = "hey sonny"

generator = pipeline(WAKE_WORD, voice='af_heart', speed=1.0)
for gs, ps, audio in generator:
    display(Audio(data=audio, rate=24000, autoplay=True))
    sf.write(f"{OUTPUT_DIR}/test_sample.wav", audio, 24000)
    print(f"Duration: {len(audio)/24000:.2f}s | phonemes: {ps}")

A starting subset of English voices [VOICES.md](https://huggingface.co/hexgrad/Kokoro-82M/blob/main/VOICES.md)


In [ ]:
# American English voices
A_VOICES = [
    "af_heart", "af_alloy", "af_aoede", "af_bella", "af_jessica",
    "af_kore", "af_nicole", "af_nova", "af_river",
    "af_sarah", "af_sky", "am_adam", "am_echo", "am_eric", "am_fenrir",
    "am_liam", "am_michael", "am_onyx", "am_puck", "am_santa"
]
B_VOICES = [
    "bf_alice", "bf_emma", "bf_isabella", "bf_lily", "bm_daniel",
    "bm_fable", "bm_george", "bm_lewis"
]

VOICES = A_VOICES + B_VOICES
SPEEDS = [0.85, 1.0, 1.15]

print(f"{len(VOICES)} voices x {len(SPEEDS)} speeds = {len(VOICES)*len(SPEEDS)} positive samples for one phrase")

Generating positives batch

In [ ]:
records = []
idx = 0
for voice in VOICES:
    for speed in SPEEDS:
        generator = pipeline(WAKE_WORD, voice=voice, speed=speed)
        for gs, ps, audio in generator:
            path = f"{OUTPUT_DIR}/positive/{idx:04d}_{voice}_{speed}.wav"
            sf.write(path, audio, 24000)
            records.append({
                "path": path, "voice": voice, "speed": speed,
                "duration_s": len(audio) / 24000,
            })
            idx += 1

df = pd.DataFrame(records)
df.describe()

Sanity check of generated samples on normal speed

In [ ]:
def listen_to_generated_samples(df: pd.DataFrame) -> None:
  for row in df.itertuples():
    if row.speed == 1.0:
      print(f"Voice: {row.voice}; Speed: {row.speed}; Duration:{row.duration_s:.2f}s")
      data, sr = sf.read(row.path)
      display(Audio(data=data, rate=sr))

In [ ]:
listen_to_generated_samples(df)

Visual Inspection: waveform + spectogram

In [ ]:
items_path = df.sample(5)["path"].tolist()

fig, axes = plt.subplots(len(items_path), 2, figsize=(10, 3 * len(items_path)))
for row_ax, path in zip(axes, items_path):
    y, sr = librosa.load(path, sr=None)
    row_ax[0].plot(y)
    row_ax[0].set_title(f"Waveform: {os.path.basename(path)}")
    S = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    librosa.display.specshow(S, sr=sr, x_axis="time", y_axis="hz", ax=row_ax[1])
    row_ax[1].set_title("Spectrogram")
plt.tight_layout()
plt.show()

Testing negative samples

In [ ]:
NEGATIVE_PHRASES = [
    "hello", "good morning", "what time is it",
    "play some music", "turn off the lights",
    "hey siri", "okay google", "alexa",
]

negative_records = []
neg_idx = 0
for phrase in NEGATIVE_PHRASES:
    for voice in VOICES[:6]: # explore only subset of voices
        generator = pipeline(phrase, voice=voice, speed=1.0)
        for gs, ps, audio in generator:
            path = f"{OUTPUT_DIR}/negative/{neg_idx:04d}_{voice}.wav"
            sf.write(path, audio, 24000)
            negative_records.append({
                "path": path, "voice": voice, "speed": 1.0,
                "duration_s": len(audio) / 24000,
            })
            neg_idx += 1

neg_df = pd.DataFrame(negative_records)
neg_df.describe()

In [ ]:
listen_to_generated_samples(neg_df.sample(10))